# Task 4: Time Series — Total Installs Trend by Category, with Growth Shading

**Requirements:**
- Line chart of the trend of **total installs over time**, segmented by **app category**
- **Shade** the area under the curve wherever month-over-month installs growth exceeds **20%**
- Filters:
  - App name must **not start with** `x`, `y`, or `z`
  - Category must **start with** `E`, `C`, or `B`
  - App name must **not contain** the letter `s`
  - Reviews > 500
- Category label translation on the chart: **Beauty → Hindi**, **Business → Tamil**, **Dating → German**
- Display rule: only visible between **6 PM – 9 PM IST**

### Two data notes, flagged up front

1. **The dataset has no real historical time series** — `Installs` is a single snapshot count per app, not installs-by-date. To show a genuine "trend over time," we treat each app's `Last Updated` date as when its installs entered the data, bucket by month, and take the **cumulative sum per category** over time. This is a standard workaround for this dataset, not real historical install data, and is called out again below.
2. **Category filter vs. translation instruction conflict:** the category filter says "starts with E, C, or B," which does **not** include `DATING` (starts with D). The brief separately asks to translate the Dating category into German. Since `DATING` cannot pass the E/C/B filter, it will never appear on this chart — so its German translation is defined below but has nothing to render. `BEAUTY` and `BUSINESS` do start with B, so their Hindi/Tamil translations are applied and visible.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

## 1. Load and clean the data

In [2]:
df = pd.read_csv('googleplaystore.csv')
df = df.drop_duplicates(subset='App', keep='first')
df = df[~df['Category'].astype(str).str.contains(r'^\d', regex=True, na=False)]

df['Installs_Num'] = pd.to_numeric(
    df['Installs'].astype(str).str.replace(',', '', regex=False).str.replace('+', '', regex=False),
    errors='coerce'
)
df['Reviews_Num'] = pd.to_numeric(df['Reviews'], errors='coerce')
df['Last_Updated_Date'] = pd.to_datetime(df['Last Updated'], errors='coerce')

df[['App','Category','Installs_Num','Reviews_Num','Last_Updated_Date']].head()

,App,Category,Installs_Num,Reviews_Num,Last_Updated_Date
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,10000,159,2018-01-07
1,Coloring book moana,ART_AND_DESIGN,500000,967,2018-01-15
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,5000000,87510,2018-08-01
3,Sketch - Draw & Paint,ART_AND_DESIGN,50000000,215644,2018-06-08
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,100000,967,2018-06-20


## 2. Apply filters

In [3]:
cats_eCB = [c for c in df['Category'].unique() if str(c)[0] in ('E', 'C', 'B')]

name_mask = (
    ~df['App'].astype(str).str.lower().str.startswith(('x', 'y', 'z')) &
    ~df['App'].astype(str).str.contains('s', case=False, na=False)
)

mask = (
    df['Category'].isin(cats_eCB) &
    (df['Reviews_Num'] > 500) &
    name_mask &
    df['Last_Updated_Date'].notna()
)

filtered = df[mask].copy()
print(f"Rows after filtering: {len(filtered)}")
filtered['Category'].value_counts()

Rows after filtering: 198


Category
COMMUNICATION          62
BOOKS_AND_REFERENCE    38
ENTERTAINMENT          37
EDUCATION              25
BUSINESS               22
COMICS                  6
BEAUTY                  4
EVENTS                  4
Name: count, dtype: int64

## 3. Build the monthly cumulative installs trend per category

In [4]:
filtered['Month'] = filtered['Last_Updated_Date'].dt.to_period('M').dt.to_timestamp()

monthly = filtered.groupby(['Category', 'Month'])['Installs_Num'].sum().reset_index()

full_range = pd.date_range(monthly['Month'].min(), monthly['Month'].max(), freq='MS')
pivot = monthly.pivot(index='Month', columns='Category', values='Installs_Num').reindex(full_range).fillna(0)
pivot.index.name = 'Month'

cumulative = pivot.cumsum()
cumulative.tail()

Category,BEAUTY,BOOKS_AND_REFERENCE,BUSINESS,COMICS,COMMUNICATION,EDUCATION,ENTERTAINMENT,EVENTS
Month,,,,,,,,
2018-04-01,5000.0,30020000.0,4711000.0,100000.0,5.856000e+07,8210000.0,51000000.0,0.0
2018-05-01,5000.0,70620000.0,4811000.0,200000.0,6.356000e+07,10310000.0,57000000.0,10000.0
2018-06-01,1005000.0,117720000.0,20011000.0,200000.0,7.466000e+07,10310000.0,78100000.0,510000.0
2018-07-01,2005000.0,219320000.0,31611000.0,6200000.0,3.903900e+08,39910000.0,458900000.0,1610000.0
2018-08-01,2105000.0,329370000.0,146711000.0,11250000.0,1.688900e+09,41910000.0,526100000.0,1610000.0


## 4. Detect month-over-month growth spikes (>20%)

In [5]:
pct_change = cumulative.pct_change().replace([np.inf, -np.inf], np.nan)
growth_flags = pct_change > 0.20

print('Number of >20% MoM growth months, by category:')
print(growth_flags.sum().sort_values(ascending=False))

Number of >20% MoM growth months, by category:
Category
BUSINESS               9
EDUCATION              9
COMMUNICATION          7
BOOKS_AND_REFERENCE    7
ENTERTAINMENT          6
COMICS                 3
BEAUTY                 2
EVENTS                 2
dtype: int64


## 5. Category label translation (Beauty → Hindi, Business → Tamil, Dating → German)

In [6]:
category_translations = {
    'BEAUTY': 'सौंदर्य (Beauty)',      # Hindi for Beauty
    'BUSINESS': 'வணிகம் (Business)',   # Tamil for Business
    'DATING': 'Partnersuche (Dating)'  # German for Dating -- not shown, see note above
}

def display_label(cat):
    return category_translations.get(cat, cat)

{c: display_label(c) for c in cumulative.columns}

{'BEAUTY': 'सौंदर्य (Beauty)',
 'BOOKS_AND_REFERENCE': 'BOOKS_AND_REFERENCE',
 'BUSINESS': 'வணிகம் (Business)',
 'COMICS': 'COMICS',
 'COMMUNICATION': 'COMMUNICATION',
 'EDUCATION': 'EDUCATION',
 'ENTERTAINMENT': 'ENTERTAINMENT',
 'EVENTS': 'EVENTS'}

## 6. Plot: time series line chart with shaded growth periods

In [7]:
palette = ['#4C9AFF', '#FF8C42', '#3DDC97', '#FF6B6B', '#B084F5', '#FFD166', '#06D6A0', '#EF476F']

fig = go.Figure()

for i, cat in enumerate(cumulative.columns):
    color = palette[i % len(palette)]
    y = cumulative[cat]
    x = cumulative.index
    label = display_label(cat)

    # base line
    fig.add_trace(go.Scatter(
        x=x, y=y, mode='lines+markers', name=label,
        line=dict(color=color, width=2)
    ))

    # shaded segments where MoM growth > 20%
    flags = growth_flags[cat]
    for j in range(1, len(x)):
        if flags.iloc[j]:
            fig.add_trace(go.Scatter(
                x=[x[j-1], x[j], x[j], x[j-1]],
                y=[0, 0, y.iloc[j], y.iloc[j-1]],
                fill='toself',
                fillcolor=color,
                opacity=0.18,
                line=dict(width=0),
                showlegend=False,
                hoverinfo='skip'
            ))

fig.update_layout(
    title='Cumulative Installs Trend by Category<br><sup>Shaded = month-over-month growth > 20%</sup>',
    xaxis_title='Month',
    yaxis_title='Cumulative Installs',
    legend=dict(orientation='h', y=1.18, x=0.5, xanchor='center'),
    height=620,
    template='plotly_white',
    margin=dict(t=120, b=60)
)

fig.show()

## 7. Export for the combined dashboard

In [8]:
cumulative.to_csv('task4_cumulative_installs.csv')
growth_flags.to_csv('task4_growth_flags.csv')
print('Saved: task4_cumulative_installs.csv, task4_growth_flags.csv')

Saved: task4_cumulative_installs.csv, task4_growth_flags.csv


### Note on the 6PM–9PM IST display rule
Same pattern as Tasks 1–3: dashboard-level display rule, implemented with the same IST time-check logic in the combined `dashboard.html`.